# Question 2 — PCB defect inspection

**Inputs:** `PCB_Reference.jpg` and `PCB_Test.jpg`.

Both images are resized to 512 × 512 using bilinear interpolation, median-filtered with a 3 × 3 window and zero padding, and segmented with iterative global thresholding (stop when the threshold change is below 0.5). A 3 × 3 morphological opening is then applied.

Defects are found by comparing the processed binary images:

- Missing conductive regions = Reference foreground but not Test foreground.
- Unwanted conductive regions = Test foreground but not Reference foreground.

Connected components use 8-connectivity. Both implementations are included.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

REFERENCE_PATH = "PCB_Reference.jpg"
TEST_PATH = "PCB_Test.jpg"


def show_img(title, im, cmap="gray"):
    plt.figure(figsize=(7, 5))
    plt.imshow(im, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()


def read_gray(path):
    a = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if a is None:
        raise FileNotFoundError(f"Cannot read {path}")
    return a

## Style 1 — OpenCV built-in methods


In [ ]:
ref1, test1 = read_gray(REFERENCE_PATH), read_gray(TEST_PATH)
ref1 = cv2.resize(ref1, (512, 512), interpolation=cv2.INTER_LINEAR)
test1 = cv2.resize(test1, (512, 512), interpolation=cv2.INTER_LINEAR)

ref_med1 = cv2.medianBlur(
    cv2.copyMakeBorder(ref1, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0), 3
)[1:-1, 1:-1]
test_med1 = cv2.medianBlur(
    cv2.copyMakeBorder(test1, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0), 3
)[1:-1, 1:-1]


def iterative_threshold_cv(gray, tol=0.5, max_iter=1000):
    t = (float(gray.min()) + float(gray.max())) / 2
    for _ in range(max_iter):
        low = gray[gray <= t]
        high = gray[gray > t]
        if len(low) == 0 or len(high) == 0:
            break
        new_t = (float(low.mean()) + float(high.mean())) / 2
        if abs(new_t - t) < tol:
            t = new_t
            break
        t = new_t
    return t, (gray > t).astype(np.uint8) * 255


thr_ref1, ref_bin1 = iterative_threshold_cv(ref_med1)
thr_test1, test_bin1 = iterative_threshold_cv(test_med1)

kernel = np.ones((3, 3), np.uint8)
ref_open1 = cv2.morphologyEx(
    ref_bin1, cv2.MORPH_OPEN, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=0
)
test_open1 = cv2.morphologyEx(
    test_bin1, cv2.MORPH_OPEN, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=0
)

missing1 = (ref_open1 > 0) & (test_open1 == 0)
unwanted1 = (test_open1 > 0) & (ref_open1 == 0)


def component_stats_cv(mask):
    n, lab, stats, cent = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    comps = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        comps.append(
            {
                "label": i,
                "area": int(area),
                "centroid": tuple(cent[i]),
                "bbox": (int(x), int(y), int(w), int(h)),
            }
        )
    return lab, comps


missing_labels1, missing_comps1 = component_stats_cv(missing1)
unwanted_labels1, unwanted_comps1 = component_stats_cv(unwanted1)
print(f"Reference threshold: {thr_ref1:.2f}; Test threshold: {thr_test1:.2f}")
print("Missing conductive-region components:", len(missing_comps1))
print("Unwanted conductive-region components:", len(unwanted_comps1))
print("Total detected defects:", len(missing_comps1) + len(unwanted_comps1))

overlay1 = np.repeat(test1[:, :, None], 3, axis=2)
overlay1[missing1] = [255, 0, 0]  # red
overlay1[unwanted1] = [0, 255, 0]  # green
show_img("Reference — opened binary", ref_open1)
show_img("Test — opened binary", test_open1)
plt.figure(figsize=(8, 6))
plt.imshow(overlay1)
plt.title("PCB defects: missing=red, unwanted=green")
plt.axis("off")
plt.show()

## Style 2 — NumPy implementation (OpenCV only for `imread`)

No OpenCV resize, filtering, thresholding, morphology or labeling calls are used below. Bilinear interpolation, zero-padded median filtering, iterative thresholding, opening and 8-connected components are implemented with NumPy/Python.


In [ ]:
# cv2.imread is the ONLY OpenCV operation in this section.
ref2 = cv2.imread(REFERENCE_PATH, 0)
test2 = cv2.imread(TEST_PATH, 0)
if ref2 is None or test2 is None:
    raise FileNotFoundError("Place both PCB input images beside this notebook.")
ref2 = ref2.astype(np.uint8)
test2 = test2.astype(np.uint8)


def resize_bilinear(a, out_h=512, out_w=512):
    a = a.astype(float)
    h, w = a.shape
    ys = np.linspace(0, h - 1, out_h)
    xs = np.linspace(0, w - 1, out_w)
    y0 = np.floor(ys).astype(int)
    x0 = np.floor(xs).astype(int)
    y1 = np.minimum(y0 + 1, h - 1)
    x1 = np.minimum(x0 + 1, w - 1)
    dy = ys - y0
    dx = xs - x0
    top = (1 - dx)[None, :] * a[y0[:, None], x0[None, :]] + dx[None, :] * a[
        y0[:, None], x1[None, :]
    ]
    bot = (1 - dx)[None, :] * a[y1[:, None], x0[None, :]] + dx[None, :] * a[
        y1[:, None], x1[None, :]
    ]
    return (
        np.rint((1 - dy)[:, None] * top + dy[:, None] * bot)
        .clip(0, 255)
        .astype(np.uint8)
    )


def median3_zero(a):
    p = np.pad(a, 1, mode="constant", constant_values=0)
    windows = np.stack(
        [p[i : i + a.shape[0], j : j + a.shape[1]] for i in range(3) for j in range(3)],
        axis=0,
    )
    return np.median(windows, axis=0).astype(np.uint8)


def iterative_threshold_np(a, tol=0.5, max_iter=1000):
    t = (float(a.min()) + float(a.max())) / 2
    for _ in range(max_iter):
        lo = a[a <= t]
        hi = a[a > t]
        if lo.size == 0 or hi.size == 0:
            break
        nt = (float(lo.mean()) + float(hi.mean())) / 2
        if abs(nt - t) < tol:
            t = nt
            break
        t = nt
    return t, a > t


def dilate3(mask):
    p = np.pad(mask, 1, mode="constant", constant_values=False)
    out = np.zeros(mask.shape, bool)
    for i in range(3):
        for j in range(3):
            out |= p[i : i + mask.shape[0], j : j + mask.shape[1]]
    return out


def erode3(mask):
    p = np.pad(mask, 1, mode="constant", constant_values=False)
    out = np.ones(mask.shape, bool)
    for i in range(3):
        for j in range(3):
            out &= p[i : i + mask.shape[0], j : j + mask.shape[1]]
    return out


ref2r = resize_bilinear(ref2)
test2r = resize_bilinear(test2)
ref2m = median3_zero(ref2r)
test2m = median3_zero(test2r)
thr_ref2, ref2b = iterative_threshold_np(ref2m)
thr_test2, test2b = iterative_threshold_np(test2m)
ref2o = erode3(dilate3(ref2b))
test2o = erode3(dilate3(test2b))

missing2 = ref2o & ~test2o
unwanted2 = test2o & ~ref2o

In [ ]:
def components8(mask):
    h, w = mask.shape
    labels = np.zeros((h, w), np.int32)
    comps = []
    lab = 0
    neigh = [(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]
    for y in range(h):
        for x in range(w):
            if mask[y, x] and labels[y, x] == 0:
                lab += 1
                stack = [(y, x)]
                labels[y, x] = lab
                pix = []
                while stack:
                    cy, cx = stack.pop()
                    pix.append((cy, cx))
                    for dy, dx in neigh:
                        ny, nx = cy + dy, cx + dx
                        if (
                            0 <= ny < h
                            and 0 <= nx < w
                            and mask[ny, nx]
                            and labels[ny, nx] == 0
                        ):
                            labels[ny, nx] = lab
                            stack.append((ny, nx))
                yy = np.fromiter((p[0] for p in pix), int)
                xx = np.fromiter((p[1] for p in pix), int)
                comps.append(
                    {
                        "label": lab,
                        "area": len(pix),
                        "centroid": (float(xx.mean()), float(yy.mean())),
                        "bbox": (
                            int(xx.min()),
                            int(yy.min()),
                            int(xx.max() - xx.min() + 1),
                            int(yy.max() - yy.min() + 1),
                        ),
                    }
                )
    return labels, comps


missing_labels2, missing_comps2 = components8(missing2)
unwanted_labels2, unwanted_comps2 = components8(unwanted2)
print(f"Reference threshold: {thr_ref2:.2f}; Test threshold: {thr_test2:.2f}")
print("Missing conductive-region components:", len(missing_comps2))
print("Unwanted conductive-region components:", len(unwanted_comps2))
print("Total detected defects:", len(missing_comps2) + len(unwanted_comps2))
print("Missing component details:", missing_comps2)
print("Unwanted component details:", unwanted_comps2)

overlay2 = np.repeat(test2r[:, :, None], 3, axis=2)
overlay2[missing2] = [255, 0, 0]
overlay2[unwanted2] = [0, 255, 0]
show_img("Reference — opened binary", ref2o.astype(np.uint8) * 255)
show_img("Test — opened binary", test2o.astype(np.uint8) * 255)
plt.figure(figsize=(8, 6))
plt.imshow(overlay2)
plt.title("PCB defects: missing=red, unwanted=green")
plt.axis("off")
plt.show()

### Notes

- Component counts are counts of connected defect regions, not counts of individual defect pixels.
- The question's sample counts are tied to its particular PCB images. Supply the exact `PCB_Reference.jpg` and `PCB_Test.jpg` to reproduce comparable results.
- A threshold comparison uses `gray > threshold` as foreground. If your lab convention treats dark PCB traces as foreground, invert both binary masks consistently before calculating the difference masks.
